# M1.3 · Regularization: ridge ve lasso

Ağırlıkları küçük tutmanın iki yolu. 2×2'lik bir ridge sistemini elle çözüp soft-threshold'u uyguluyor, saf Python'da coordinate descent ile lasso yazıyor, NumPy'da tüm regularization yolunu tek SVD'den çıkarıyor, PyTorch'ta `weight_decay`'in L2 cezasıyla ilişkisini kuruyor ve en sonda k-fold cross-validation'ı λ'lar üzerinde döngüsüz yapıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m1-3-regularization.html](https://m-gonul.github.io/pytorch-lab/m1-3-regularization.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m1_3_regularization.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref_lasso(X, y, lam, iters):\n    n, d = len(X), len(X[0])\n    w = [0.0] * d\n    for _ in range(iters):\n        for j in range(d):\n            rho = sum(X[i][j] * (y[i] - sum(X[i][k] * w[k] for k in range(d) if k != j)) for i in range(n)) / n\n            z = sum(X[i][j] ** 2 for i in range(n)) / n\n            s = rho - lam if rho > lam else (rho + lam if rho < -lam else 0.0)\n            w[j] = s / z\n    return w\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    R.test(\"soft_threshold\", lambda: close([need(ns, \"soft_threshold\")(z, 0.5) for z in (1.5, -0.3, -2, 0.5, -0.5)], [1, 0, -1.5, 0, 0]))\n    X, y = [[1, 0], [0, 1], [1, 1]], [1, 2, 3]\n    R.test(\"ridge2: OLS (1, 2) ve λ = 1 (0.875, 1.375)\", lambda: (close(need(ns, \"ridge2\")(X, y, 0.0), [1, 2]), close(need(ns, \"ridge2\")(X, y, 1.0), [0.875, 1.375])))\n    rng = random.Random(0)\n    Xr = [[rng.gauss(0, 1) for _ in range(4)] for _ in range(30)]\n    wt = [2.0, 0.0, -1.0, 0.0]\n    yr = [sum(a * b for a, b in zip(x, wt)) + rng.gauss(0, 0.1) for x in Xr]\n\n    def lc():\n        w = need(ns, \"lasso_cd\")(Xr, yr, 0.1, 50)\n        close(w, ref_lasso(Xr, yr, 0.1, 50), rtol=1e-6, atol=1e-9)\n        return \"katsayılar: \" + \", \".join(f\"{v:.3f}\" for v in w)\n    R.test(\"lasso_cd referans coordinate descent ile aynı\", lc)\n\n    def sparse():\n        w = need(ns, \"lasso_cd\")(Xr, yr, 0.3, 50)\n        zeros = sum(v == 0.0 for v in w)\n        ensure(zeros >= 2, f\"λ = 0.3'te yalnız {zeros} katsayı tam sıfır; gerçek modelde 2 sıfır var\")\n        w_big = need(ns, \"lasso_cd\")(Xr, yr, 100.0, 5)\n        ensure(all(v == 0.0 for v in w_big), \"çok büyük λ'da tüm katsayılar tam 0 olmalı\")\n        return f\"λ = 0.3 → {zeros} tam sıfır; gerçek ağırlıklar [2, 0, −1, 0]\"\n    R.test(\"lasso tam sıfırlar üretiyor\", sparse)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(1)\n    X = rng.normal(size=(60, 6)) * np.array([1, 2, 0.5, 1, 3, 1])\n    y = X @ rng.normal(size=6) + 2.0 + 0.1 * rng.normal(size=60)\n    Xc, yc = X - X.mean(0), y - y.mean()\n\n    def rd():\n        for lam in (0.0, 0.5, 20.0):\n            w, b = need(ns, \"ridge\")(X, y, lam)\n            want = np.linalg.solve(Xc.T @ Xc + lam * np.eye(6), Xc.T @ yc)\n            close(w, want, what=f\"λ = {lam} w\"); close(b, y.mean() - X.mean(0) @ want, what=f\"λ = {lam} b\")\n        w0, b0 = need(ns, \"ridge\")(np.array([[1., 0], [0, 1], [1, 1]]), np.array([1., 2, 3]), 0.0)\n    R.test(\"ridge: bias'lı kapalı form (λ = 0, 0.5, 20)\", rd)\n\n    def rp():\n        lams = np.array([0.0, 0.1, 1.0, 10.0, 1e4])\n        P = need(ns, \"ridge_path\")(X, y, lams)\n        ensure(np.asarray(P).shape == (5, 6), f\"shape {np.asarray(P).shape}\")\n        for i, lam in enumerate(lams):\n            close(P[i], np.linalg.solve(Xc.T @ Xc + lam * np.eye(6), Xc.T @ yc), rtol=1e-7, atol=1e-9, what=f\"λ = {lam}\")\n        n = np.linalg.norm(P, axis=1)\n        ensure(np.all(np.diff(n) < 0), \"‖w‖ λ arttıkça azalmalı\")\n        return \"‖w(λ)‖: \" + \", \".join(f\"{v:.3f}\" for v in n)\n    R.test(\"ridge_path tek SVD'den, her λ'da kapalı formla aynı\", rp)\n",
 "d": "import ast\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef ridge_cf(X, y, lam2):\n    n = X.shape[0]\n    Xc, yc = X - X.mean(0), y - y.mean()\n    w = torch.linalg.solve(Xc.T @ Xc / n + lam2 * torch.eye(X.shape[1], dtype=X.dtype), Xc.T @ yc / n)\n    return w, y.mean() - X.mean(0) @ w\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    torch.manual_seed(0)\n    X = torch.randn(120, 3)\n    y = X @ torch.tensor([1.0, -2.0, 0.3]) + 0.5 + 0.05 * torch.randn(120)\n    w_cf, b_cf = ridge_cf(X.double(), y.double(), 0.2)\n\n    def fr():\n        torch.manual_seed(1)\n        m = need(ns, \"fit_reg\")(X, y, 0.2)\n        close(m.weight.detach().view(-1), w_cf, atol=2e-3, rtol=0, what=\"weight\")\n        close(m.bias.detach(), b_cf.view(1), atol=2e-3, rtol=0, what=\"bias (cezasız)\")\n    R.test(\"fit_reg (λ₂ = 0.2) ridge kapalı formuyla aynı\", fr)\n\n    def l1():\n        torch.manual_seed(1)\n        m = need(ns, \"fit_reg\")(X, y, 0.0, lam1=0.5, steps=3000, lr=0.05)\n        w = m.weight.detach().view(-1)\n        ensure(abs(float(w[2])) < 0.05, f\"küçük gerçek katsayı (0.3) L1 ile bastırılmalı: {float(w[2]):.3f}\")\n        return \"L1 ağırlıklar: \" + \", \".join(f\"{float(v):.3f}\" for v in w) + \" (SGD tam 0 vermez; yalnız küçültür)\"\n    R.test(\"fit_reg L1: küçük katsayıyı bastırıyor\", l1)\n\n    def wd():\n        tree = ast.parse(src)\n        fn = next((n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name == \"fit_weight_decay\"), None)\n        ensure(fn is not None, \"`fit_weight_decay` bulunamadı\")\n        names = {n.attr for n in ast.walk(fn) if isinstance(n, ast.Attribute)}\n        ensure(\"abs\" not in names, \"fit_weight_decay cezayı kayba eklememeli\")\n        txt = ast.unparse(fn)\n        ensure(\"weight_decay\" in txt, \"weight_decay kullanılmıyor\")\n        torch.manual_seed(1)\n        m = need(ns, \"fit_weight_decay\")(X, y, 0.2)\n        close(m.weight.detach().view(-1), w_cf, atol=2e-3, rtol=0, what=\"weight\")\n        close(m.bias.detach(), b_cf.view(1), atol=2e-3, rtol=0, what=\"bias (decay'siz olmalı)\")\n        return \"weight_decay = 2λ ve bias muaf\"\n    R.test(\"fit_weight_decay aynı ridge çözümüne ulaşıyor\", wd)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef solve_one(X, y, lam):\n    xm, ym = X.mean(0), y.mean()\n    Xc = X - xm\n    w = torch.linalg.solve(Xc.T @ Xc + lam * torch.eye(X.shape[1], dtype=X.dtype), Xc.T @ (y - ym))\n    return w, ym - xm @ w\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"ridge_path_torch: döngü yok\", lambda: max_loops(src, 0, func=\"ridge_path_torch\"))\n    R.test(\"cv_ridge: yalnız fold döngüsü\", lambda: max_loops(src, 1, func=\"cv_ridge\", why=\"λ'lar üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n    X = torch.randn(150, 20, generator=g, dtype=torch.float64)\n    y = X[:, :4] @ torch.tensor([1.0, -1.0, 2.0, 0.5], dtype=torch.float64) + 3 + torch.randn(150, generator=g, dtype=torch.float64)\n    lams = torch.logspace(-3, 3, 30, dtype=torch.float64)\n\n    def path():\n        W, b = need(ns, \"ridge_path_torch\")(X, y, lams)\n        ensure(tuple(W.shape) == (30, 20) and tuple(b.shape) == (30,), f\"shape'ler {tuple(W.shape)}, {tuple(b.shape)}\")\n        for i in (0, 10, 29):\n            w1, b1 = solve_one(X, y, float(lams[i]))\n            close(W[i], w1, rtol=1e-7, atol=1e-9, what=f\"λ = {float(lams[i]):.3g} w\"); close(b[i], b1, rtol=1e-7, what=\"b\")\n    R.test(\"ridge_path_torch: her λ'da kapalı form\", path)\n\n    def cv():\n        mse, best = need(ns, \"cv_ridge\")(X, y, lams, k=5)\n        ref = torch.zeros(30, dtype=torch.float64)\n        idx = torch.arange(150)\n        for i in range(5):\n            va, tr = idx[i::5], idx[idx % 5 != i]\n            for j, lam in enumerate(lams):\n                w, b = solve_one(X[tr], y[tr], float(lam))\n                ref[j] += ((X[va] @ w + b - y[va]) ** 2).mean() / 5\n        close(mse, ref, rtol=1e-7)\n        close(best, lams[ref.argmin()])\n        return f\"en iyi λ = {float(best):.3g}, CV MSE {float(mse.min()):.3f} (λ → 0: {float(mse[0]):.3f}, λ → ∞: {float(mse[-1]):.3f})\"\n    R.test(\"cv_ridge: referans CV ile aynı\", cv)\n    if \"ridge_path_torch\" not in ns:\n        return\n    Xb = torch.randn(1000, 100, generator=g, dtype=torch.float64)\n    yb = Xb @ torch.randn(100, generator=g, dtype=torch.float64)\n    lb = torch.logspace(-3, 3, 200, dtype=torch.float64)\n    t_loop = bench(lambda: [solve_one(Xb, yb, float(l)) for l in lb], repeat=3, number=1)\n    t_svd = bench(lambda: ns[\"ridge_path_torch\"](Xb, yb, lb), repeat=3)\n    R.table(\"200 λ, n = 1000, d = 100\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for λ: solve (200 × O(d³))\", fmt_time(t_loop), \"1×\"],\n             [\"senin ridge_path_torch (1 SVD)\", fmt_time(t_svd), f\"{t_loop / t_svd:,.0f}× hızlı\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m1_3_regularization as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Özellik sayısı örnek sayısına yaklaşınca ya da özellikler birbiriyle ilişkiliyken en küçük kareler büyük, kararsız katsayılar üretir: aşırı uyum. Çare, katsayıların büyüklüğünü kayba eklemek:

$$
\text{Ridge: }\ \min_w \lVert y-Xw\rVert^2+\lambda\lVert w\rVert_2^2,\qquad
\text{Lasso: }\ \min_w \frac1{2n}\lVert y-Xw\rVert^2+\lambda\lVert w\rVert_1 .
$$

Ridge'in kapalı formu var: gradyanı sıfıra eşitle,

$$
(X^\top X+\lambda I)\,w=X^\top y .
$$

$\lambda I$ köşegeni büyütür; sistem her zaman terslenebilir olur. Lasso'nun kapalı formu yok ama tek bir koordinatta çözüm basit: diğerleri sabitken $w_j$ için optimum bir soft-threshold:

$$
w_j \leftarrow \frac{S\!\left(\tfrac1n x_j^\top r_j,\ \lambda\right)}{\tfrac1n\lVert x_j\rVert^2},\qquad r_j=y-\sum_{k\ne j}x_kw_k .
$$

Bu adımı koordinatlar üzerinde tekrarlamak **coordinate descent**; glmnet ve scikit-learn'ün `Lasso`'su bunu yapar. Bias (intercept) cezalandırılmaz: veriyi ortalayıp sonra $b=\bar y-\bar x^\top w$ ile geri koyarız.

## 1 · Elle

$X^\top X$ ve $X^\top y$'yi hesapla, sonra iki $2\times2$ sistemi çöz. Ridge katsayıları OLS'ye göre **küçüldü**: $\lVert w\rVert^2$ 5'ten $2.66$'ya indi. $\lambda\to\infty$ iken tüm katsayılar sıfıra gider.

Soft-threshold'da üç durumu gör: büyük pozitif değer $t$ kadar küçülür, küçük değer **tam sıfır** olur, negatif değer işaretini koruyarak küçülür. Lasso'nun özellik seçmesinin tek mekanizması bu sıfırlama.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$X=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix}$, $y=[1,2,3]$, bias yok. Önce $X^\top X$ ve $X^\top y$**

- `a11`: $(X^\top X)_{11}$
- `a12`: $(X^\top X)_{12}$
- `xy1`: $(X^\top y)_1$
- `xy2`: $(X^\top y)_2$

**Çözümler: OLS ($\lambda=0$) ve ridge ($\lambda=1$); $2\times2$ ters: $\frac1{ad-bc}\begin{bmatrix}d&-b\\-c&a\end{bmatrix}$**

- `o1`: OLS $w_1$
- `o2`: OLS $w_2$
- `r1`: Ridge $w_1$
- `r2`: Ridge $w_2$

**Soft-threshold $S(z,t)=\text{sign}(z)\max(|z|-t,0)$, $t=0.5$**

- `s1`: $S(1.5,\,0.5)$
- `s2`: $S(-0.3,\,0.5)$
- `s3`: $S(-2,\,0.5)$

In [ ]:
cevaplar = dict(
    a11=None,
    a12=None,
    xy1=None,
    xy2=None,
    o1=None,
    o2=None,
    r1=None,
    r2=None,
    s1=None,
    s2=None,
    s3=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"a11":[2,0],"a12":[1,0],"xy1":[4,0],"xy2":[5,0],"o1":[1,0.001],"o2":[2,0.001],"r1":[0.875,0.002],"r2":[1.375,0.002],"s1":[1,0.001],"s2":[0,0.001],"s3":[-1.5,0.001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, iç içe listelerle. `numpy`/`torch` yasak.

- `soft_threshold(z, t)`
- `ridge2(X, y, lam)` — **iki** özellik, bias yok: $X^\top X+\lambda I$'yı kur, $2\times2$ ters formülüyle çöz, `[w1, w2]` döndür.
- `lasso_cd(X, y, lam, iters=100)` — herhangi sayıda özellik, bias yok, amaç $\frac1{2n}\lVert y-Xw\rVert^2+\lambda\lVert w\rVert_1$. Her iterasyonda $j=0..d-1$ sırasıyla tek koordinat güncellemesi (intro'daki formül). `w` sıfırdan başlasın.

Kontrol: büyük λ'da tüm katsayılar **tam** 0, λ = 0'da OLS'ye yakınsama.

In [ ]:
%%writefile stage_b.py
def soft_threshold(z, t):
    ...


def ridge2(X, y, lam):
    """İki özellik, bias yok. → [w1, w2]"""
    ...


def lasso_cd(X, y, lam, iters=100):
    """Amaç: (1/2n)‖y − Xw‖² + λ‖w‖₁, bias yok. → w (liste)"""
    ...


X = [[1, 0], [0, 1], [1, 1]]
y = [1, 2, 3]
print(ridge2(X, y, 0.0), ridge2(X, y, 1.0))     # [1, 2] [0.875, 1.375]
print(soft_threshold(1.5, 0.5), soft_threshold(-0.3, 0.5))
print(lasso_cd(X, y, 0.3))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da bias'lı ridge ve tüm λ'lar için yol.

- `ridge(X, y, lam)` → `(w, b)`. Önce `X` ve `y`'yi ortala, $(X_c^\top X_c+\lambda I)w=X_c^\top y_c$'yi `np.linalg.solve` ile çöz, $b=\bar y-\bar x^\top w$. **Döngü yok.**
- `ridge_path(X, y, lams)` → `(L, d)`; her satır bir λ'nın `w`'si. Ortalanmış $X_c=U\,\text{diag}(s)\,V^\top$ (ince SVD) için

$$
w(\lambda)=V\,\text{diag}\!\left(\frac{s}{s^2+\lambda}\right)U^\top y_c .
$$

SVD bir kez alınır; λ'lar broadcasting ile tek ifadede: `s / (s**2 + lams[:, None])` → `(L, k)`. **Döngü yok.**

In [ ]:
%%writefile stage_c.py
import numpy as np


def ridge(X, y, lam):
    """→ (w, b); bias cezalandırılmaz. Döngü yok."""
    ...


def ridge_path(X, y, lams):
    """→ (L, d): her λ için w; tek SVD. Döngü yok."""
    ...


rng = np.random.default_rng(0)
X = rng.normal(size=(50, 5))
y = X @ np.array([1.0, 0, -2, 0, 0.5]) + 3 + 0.1 * rng.normal(size=50)
print(ridge(X, y, 1.0))
print(ridge_path(X, y, np.array([0.0, 1.0, 100.0])))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta gradyanla aynı amaç: $\text{MSE}+\lambda_2\lVert w\rVert^2+\lambda_1\lVert w\rVert_1$.

- `fit_reg(X, y, lam2, lam1=0.0, steps=3000, lr=0.05)` → `nn.Linear`; cezayı kayba **elle** ekle (yalnız `weight`'e, `bias`'a değil), `optim.SGD`. $\lambda_1=0$ iken sonuç ridge kapalı formuna eşit olmalı: $(\tfrac1n X_c^\top X_c+\lambda_2 I)w=\tfrac1n X_c^\top y_c$.
- `fit_weight_decay(X, y, lam2, steps=3000, lr=0.05)` — cezayı kayba **eklemeden**, `optim.SGD`'nin `weight_decay` parametresiyle aynı modeli elde et. İki şeye dikkat: doğru `weight_decay` değeri ve bias'ın decay'den muaf tutulması (param groups: `[{"params": [w], "weight_decay": …}, {"params": [b], "weight_decay": 0}]`).

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


def fit_reg(X, y, lam2, lam1=0.0, steps=3000, lr=0.05):
    """MSE + lam2‖w‖² + lam1‖w‖₁ (bias hariç), cezalar kayba elle eklenir."""
    ...


def fit_weight_decay(X, y, lam2, steps=3000, lr=0.05):
    """Aynı ridge modeli, ama ceza yerine SGD weight_decay ile."""
    ...


torch.manual_seed(0)
X = torch.randn(100, 3)
y = X @ torch.tensor([1.0, -2.0, 0.0]) + 0.5
print(fit_reg(X, y, 0.1).weight, fit_weight_decay(X, y, 0.1).weight)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

λ'yı seçmek için k-fold cross-validation: veriyi $k$ parçaya böl; her parça bir kez doğrulama, kalanlar eğitim. Her λ için ortalama doğrulama hatası; en düşüğü seç.

Döngü yapısı: `for fold` × `for λ` × (çöz + değerlendir). λ döngüsünü kaldır: her fold'da ridge yolunu (c'deki SVD formülü) **tüm λ'lar için birden** hesapla, doğrulama tahminleri `(n_val, L)` matrisi, MSE'ler `(L,)`.

- `ridge_path_torch(X, y, lams)` → `(W: (L, d), b: (L,))`; bias'lı, SVD ile, **döngü yok**.
- `cv_ridge(X, y, lams, k=5)` → `(mse: (L,), best_lam)`. Fold'lar sırayla: `i`. fold `idx[i::k]` (ön karıştırma yok, deterministik). Fold döngüsü serbest, λ döngüsü yok.

Kontrol, 200 λ için `for λ: solve` ile SVD yolunu yarıştırır.

In [ ]:
%%writefile stage_e.py
import torch


def ridge_path_torch(X, y, lams):
    """→ (W: (L, d), b: (L,)). Tek SVD, döngü yok."""
    ...


def cv_ridge(X, y, lams, k=5):
    """→ (mse: (L,), best_lam). Fold döngüsü serbest, λ döngüsü yok. i. fold: idx[i::k]."""
    ...


torch.manual_seed(0)
X = torch.randn(200, 30, dtype=torch.float64)
y = X[:, :5] @ torch.ones(5, dtype=torch.float64) + torch.randn(200, dtype=torch.float64)
lams = torch.logspace(-3, 3, 50, dtype=torch.float64)
mse, best = cv_ridge(X, y, lams)
print(best)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Tibshirani (1996): Lasso

[Tibshirani — Regression Shrinkage and Selection via the Lasso (JRSS-B 1996)](https://www.jstor.org/stable/2346178)

Lasso'yu koordinat inişiyle (yumuşak eşikleme) yaz, düzenlileştirme yolunu (λ'ya göre katsayılar, makale Şekil 2) prostate veri setinde ya da sentetik seyrek bir problemde çiz; ridge ile karşılaştır. Seyrek gerçek katsayılı (s ≪ d) problemde destek kurtarma oranını λ'ya göre ölç; makalenin ortogonal tasarımdaki kapalı form çözümünü (Denk. 3: sign(β̂)(|β̂| − γ)₊) doğrula.


**Adımlar**

1. Sıcak başlangıçla (warm start) λ yolunu büyükten küçüğe izle.

**Kabul ölçütleri**

- Yol grafikleri: Lasso katsayıları tam sıfıra iner, ridge inmez.
- Ortogonal tasarımda kapalı formla birebir sonuç.

## Kaynaklar

- [ISLP — Bölüm 6.2 Shrinkage Methods (ridge, lasso)](https://www.statlearning.com/) — doğrulanmadı
- [Elements of Statistical Learning — Bölüm 3.4](https://hastie.su.domains/ElemStatLearn/) — doğrulanmadı
- [Dive into Deep Learning §3.7 Weight Decay](https://d2l.ai/chapter_linear-regression/weight-decay.html) — CC BY-SA 4.0 / MIT-0
- [Friedman, Hastie, Tibshirani — Regularization Paths via Coordinate Descent (glmnet)](https://www.jstatsoft.org/article/view/v033i01) — CC BY 3.0